# CAI 5607 — Homework 3
## Q5. Build a Multimodal Learning Assistant — 45 points
**A 7 · B 8 · C 13 · D 7 · E 10**

Complete `student_agent.py` and the written responses below. All live execution and downloads default to off. This notebook can run in setup-only mode before the TODOs are complete, but that does **not** complete the assignment. Use the entire student folder, not only this notebook. The supplied six text sources and one toy figure are insufficient for final submission.

**Progression:** curated materials → retrieval → bounded tool use → learner practice/feedback → controlled evaluation. Reuse the finished assistant for Q6; do not build it twice.

In [ ]:
from pathlib import Path
import json, sys, subprocess
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()  # Never print key values.

from hw3_utils import (load_corpus, load_jsonl, TfidfRetriever, Limits, source_recall_at_k,
                       save_json, save_jsonl, validate_chunks)
from setup_helpers import (read_config, build_index, make_backends, make_learning_session,
                           check_case_splits, environment_record, MIN_TEXT_DOCUMENTS, MIN_VISUAL_ITEMS)
from student_agent import EvidenceAgent, parse_action, build_system_prompt
from experiment import evaluate_suite

CONFIG = read_config('config.json')
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
BUILD_INDEX = False
ALLOW_EMBEDDING_DOWNLOAD = False
RUN_STUDENT_TESTS = False
EVALUATE_RETRIEVAL = False
RUN_LIVE_DEMO = False
RUN_DEV = False
RUN_FINAL_TEST = False
print('Live calls enabled in configuration:', CONFIG['enable_live'])
print('All execution switches default to False. Fill TODOs before enabling them.')
print(json.dumps(environment_record(), indent=2))

## A. Data preparation and learner state — 7 points
Convert permitted materials into text plus actual figures; keep original locators and hashes in the manifest. Curate at least 10 substantive text documents and 5 visual items. Leave evaluation questions, answers and learner logs outside `data/`.

**Response:** Describe your corpus, permissions, preprocessing, original/generated distinction and the learner state you maintain.

TODO: Write your response.

In [ ]:
sources = load_corpus(CONFIG['manifest'])
print('Text documents:', sum(s['kind']=='text' for s in sources.values()))
print('Visual items:', sum(s['kind']=='image' for s in sources.values()))
print('Source IDs:', list(sources))
print('A demo corpus is provided for offline checks only; replace it for final evaluation.')

## Supplied software checks
The following tests use only deterministic/scripted fixtures and mocked HTTP transports. Passing them is **not** a claim of model grounding, learning impact or production security.

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'tests.test_provided', 'tests.test_approval', 'tests.test_learning_session', 'tests.test_ui_support', 'tests.test_assignment_requirements', 'tests.test_reporting'],
    text=True, capture_output=True)
print(result.stdout + result.stderr)
assert result.returncode == 0, 'A supplied utility test failed.'

## B. Chunking and retrieval — 8 points
Implement `chunk_document` and `DenseRetriever.search` in `student_agent.py`. Preserve source spans and stable IDs. Select a pretrained embedding model and revision in config.json. TF-IDF is a lexical baseline, not a pretrained semantic model. Download/load the embedding model only when you enable the flag below.

In [ ]:
chunks = lexical = dense = retriever = None
if BUILD_INDEX:
    sources, chunks, lexical, dense = build_index(CONFIG, allow_model_download=ALLOW_EMBEDDING_DOWNLOAD)
    retriever = dense if CONFIG['retriever']=='dense' else lexical
    if retriever is None:
        raise ValueError('Dense retrieval requires an explicit embedding model and download/load permission.')
    save_json(OUT/'index_metadata.json', {'configuration':CONFIG,
        'sources':{k:{f:v.get(f) for f in ('title','source','version','locator','status','sha256','generated')} for k,v in sources.items()},
        'chunk_count':len(chunks)})
    print('Built', len(chunks), 'chunks with selected retriever:', CONFIG['retriever'])
else:
    print('Index building skipped until student implementations are complete.')

In [ ]:
if EVALUATE_RETRIEVAL:
    if lexical is None or dense is None:
        raise ValueError('Build both retrievers first.')
    dev = load_jsonl('eval/dev_cases.jsonl')
    rows=[]
    for case in dev:
        # Text-source recall only; image-inspection questions are evaluated separately.
        required=[s for s in case.get('relevant_source_ids',[]) if sources[s]['kind']=='text']
        for name,r in [('lexical',lexical),('dense',dense)]:
            hits=r.search(case['question'],k=CONFIG['top_k'])
            rows.append({'case_id':case['id'],'retriever':name,'required_text_sources':required,
                         'hit_ids':[h['id'] for h in hits],
                         'recall':source_recall_at_k(hits,required,CONFIG['top_k'])})
    save_jsonl(OUT/'dev_retrieval.jsonl',rows)
    for name in ('lexical','dense'):
        valid=[r['recall'] for r in rows if r['retriever']==name and r['recall'] is not None]
        print(name, 'evaluated cases:',len(valid),'mean recall:',sum(valid)/len(valid) if valid else None)

**B response:** Report the chosen chunking and embedding model, development Recall@k, current-source filtering and one context-construction decision. State which visual details require actual image inspection.

TODO: Write your response.

## C. Agent controller — 13 points
Implement `parse_action`, `build_system_prompt`, and `EvidenceAgent.run` in `student_agent.py`. The action loop must use actual observations, not fabricated results. Keep the supplied tool registry and final statuses consistent. Use `awaiting_answer` plus a question to pause for practice. The LearningSession wrapper preserves actual turns and budget counters; design a prompt and controller that use it meaningfully.

In [ ]:
if RUN_STUDENT_TESTS:
    result = subprocess.run([sys.executable,'-m','unittest','tests.test_student'],text=True,capture_output=True)
    print(result.stdout + result.stderr)
    assert result.returncode == 0, 'Complete or fix the student implementations.'
else:
    print('Student-interface tests are intentionally skipped while TODOs remain.')

## D. Live interactions — 7 points
Configure one working model route and permissions first. Below, fill your actual learner topic, goal and prior knowledge. Demonstrate clarification, image-dependent reasoning, practice/feedback, a preview, explicit approval of the exact artifact, and a verified save. Labels and citations are not automatic proof of semantic correctness.

The examples of commands below describe the interface; they are not completed conversations. Use real messages and preserve the actual model outputs.

In [ ]:
session = None
TOPIC = ''            # Fill from the learner's actual statement.
GOAL = ''
PRIOR_KNOWLEDGE = ''
if RUN_LIVE_DEMO:
    if retriever is None: raise ValueError('Build the selected index first.')
    session=make_learning_session(CONFIG,sources,chunks,retriever,topic=TOPIC,goal=GOAL,
                                  prior_knowledge=PRIOR_KNOWLEDGE,
                                  output_dir=OUT/'learning_demo_artifacts')
    print('Live session created. No response has been generated yet.')

In [ ]:
LEARNER_MESSAGE = ''   # Enter a real message; rerun this cell for another turn.
TURN_KIND = 'message'   # 'answer' for an actual pending-practice attempt; 'request_solution' to ask for help.
IMAGE_IDS = []         # IDs from your manifest, e.g. an actual figure being discussed.
if session is not None and LEARNER_MESSAGE.strip():
    episode=session.respond(LEARNER_MESSAGE,kind=TURN_KIND,image_ids=IMAGE_IDS)
    print(json.dumps(episode['final'],indent=2,ensure_ascii=False))
    print('Host learner state:',json.dumps(session.state,indent=2,ensure_ascii=False))

In [ ]:
REVIEW_PENDING_ARTIFACT = False
if session is not None and REVIEW_PENDING_ARTIFACT:
    # Prompts the actual user; never set this from model-generated text.
    approved=session.review_and_approve()
    print('Actual user approval:',approved)
    # Continue via the next real learner turn so the model can request save_report.
    # Verify the returned artifact receipt AND the actual saved file contents.
if session is not None:
    session.save_log(OUT/'learning_demo_session.json')

**D response:** Link the four interaction demonstrations and their actual logs. Explain pending-question/attempt state, source evidence, approval of the exact content/destination and the verified artifact. Cite at least one case in which a prompt rule was insufficient or explain what failure tests you ran when none occurred.

TODO: Write your response.

## E. Controlled evaluation — 10 points
Create 5 development and 5 test cases. Select settings on development only; keep gold fields out of the model and index. Use a new output directory for every run. The code below uses the same answer model and shared initial image observation for all three conditions. Final tests do not grant report-write permissions.

In [ ]:
if RUN_DEV or RUN_FINAL_TEST:
    if retriever is None:raise ValueError('Build the selected retrieval index first.')
    dev_cases=load_jsonl('eval/dev_cases.jsonl')
    test_cases=load_jsonl('eval/test_cases.jsonl')
    check_case_splits(dev_cases,test_cases)
    if RUN_FINAL_TEST:
        assert sum(s['kind']=='text' for s in sources.values()) >= MIN_TEXT_DOCUMENTS
        assert sum(s['kind']=='image' for s in sources.values()) >= MIN_VISUAL_ITEMS
    answer_backend,vision_backend=make_backends(CONFIG)
    for label,cases,enabled in [('dev',dev_cases,RUN_DEV),('test',test_cases,RUN_FINAL_TEST)]:
        if enabled:
            stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
            run_dir=OUT/f'{label}_{stamp}'
            runs=evaluate_suite(cases,backend=answer_backend,vision_backend=vision_backend,
                retriever=retriever,chunks=chunks,sources=sources,
                system_prompt=build_system_prompt(CONFIG['domain']),agent_class=EvidenceAgent,
                parser=parse_action,output_root=run_dir,k=CONFIG['top_k'],
                limits=Limits(**CONFIG['limits']),configuration=CONFIG)
            print('Run saved to',run_dir,'; human annotation is still required.')

### Human annotation and final report
Fill the generated `*_human_annotations.jsonl` from source evidence and your prewritten criteria. The fields are intentionally null until human review. Then run `python summarize_run.py outputs/YOUR_RUN_DIRECTORY`. Do not use the model's confidence or a cited ID's existence as a factuality score.

**E response:** Report the actual three-condition results and extra work; report the lexical/dense development comparison; analyze actual errors, or state explicitly when none occurred. State no errors if that is what you observed. Summarize limitations and avoid claims of a measured human learning gain.

TODO: Write your response here. Use EVALUATION_GUIDE.md for metric definitions and the Canvas submission requirements for the required evidence.

### Completion check
Before submission: fill all TODOs; execute live demonstrations and 15 final episodes; finish annotations; include state/approval tests, authorized source data, cases, logs, configuration, dependencies and the exported notebook. Do not include secrets. Setup-only execution is not assignment completion.

## Optional Q6 — Code Generation interface and learning
Reuse your completed Q5 agent. Implement a web or desktop UI with persistent progress and include three captioned screenshots. Use it to study **Code Generation**. Read Q6 on Canvas, `UI_STARTER_GUIDE.md`, and `CODE_GENERATION_SESSION_GUIDE.md`.

**Q6A interface/progress/screenshots: 10 points · Q6B Code Generation session: 5 points · Q6C survey: 5 points.**

The optional Streamlit `ui_app.py` is a setup-only shell until its four UI TODOs and Q5 functions are completed. A plain command-line chat or rerunning notebook cells for each exchange is not the required UI. No second agent or unrestricted code-execution tool is required.
